In [1]:
import sys as __harness_sys
from pathlib import Path as __harness_Path
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\tutorials')
__harness_sys.path.insert(0, 'C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\scripts')
from _offline import enforce_offline as __guard, blocked_attempts as __blocked
__guard()
import capture_runresults_213 as __capture
__capture.install()
import agentic_systems as __harness_toolkit
assert __harness_toolkit.__version__ == "2.1.3"
assert "site-packages" in __harness_Path(__harness_toolkit.__file__).parts

# System: coordinar Agents

Los mismos tres flujos, ahora con Agents como pasos.

API **2.1.3**. Cada ejemplo muestra una entrada y su resultado. Python Runtime realiza trabajo determinista.

In [2]:
import agentic_systems as toolkit


In [3]:
@toolkit.tool
def increment(value: int) -> dict:
    """Suma uno de forma determinista."""
    return {"value": value + 1}

@toolkit.tool
def double(value: int) -> dict:
    """Duplica el valor."""
    return {"value": value * 2}

arithmetic = toolkit.skill(name="arithmetic", tools=[increment, double])
increase = toolkit.Action("increase", "increment")
multiply = toolkit.Action("multiply", "double")

def choose_next(data):
    return "multiply" if data["value"] >= 0 else "END"

def combine(branches):
    return {"value": sum(row["value"] for row in branches.values())}

In [4]:
for backend in ("native", "langgraph"):
    incrementer = toolkit.Agent("incrementer", increase, skills=(arithmetic,), backend=backend)
    doubler = toolkit.Agent("doubler", multiply, skills=(arithmetic,), backend=backend)
    sequential = toolkit.System("sequential", toolkit.Sequence((incrementer, doubler)), backend=backend)
    parallel = toolkit.System("parallel", toolkit.Parallel((incrementer, doubler), join=combine), backend=backend)
    decision = toolkit.System("decision", toolkit.Graph(
        start="increase", nodes=(("increase", incrementer), ("multiply", doubler)),
        edges=(("increase", choose_next), ("multiply", "END")),
    ), backend=backend)
    for system, expected in ((sequential, 8), (parallel, 10), (decision, 8)):
        result = system.run({"value": 3})
        assert result.ok and result.data == {"value": expected}
        print(backend, system.name, result.data)
    assert decision.run({"value": -3}).data == {"value": -2}

native sequential {'value': 8}
native parallel {'value': 10}
native decision {'value': 8}


langgraph sequential {'value': 8}
langgraph parallel {'value': 10}
langgraph decision {'value': 8}


In [5]:
__capture.save('C:\\Users\\jacob\\Documents\\AVANTECK.TEAM\\mx-m6hn-agentic_systems-feature-agentic-lab@f711b6b9bbe\\revisions\\2.1.3\\learning-evidence-elevated\\notebook-runresults\\20261023T044738543256\\07-03_system_flows.json', 'tutorials/core/03_system_flows.ipynb')
assert __blocked() == 0